# Exercício 1: Cifrando e Decifrando Arquivos com Cabeçalho Personalizado

**Objetivo:**  
Criar um programa em Python capaz de criptografar e descriptografar arquivos, utilizando AES no modo CBC e um cabeçalho de 32 bytes com metadados sobre o arquivo cifrado.

---

## Instruções

### 1. Cabeçalho do Arquivo Cifrado (32 bytes)

O início do arquivo criptografado deve conter um cabeçalho com os seguintes campos:

| Campo         | Tamanho (bytes) | Descrição                                                                 |
|---------------|-----------------|---------------------------------------------------------------------------|
| Identificador | 2               | Deve conter uma sequência fixa, ex: `b'ED'`, para indicar um arquivo cifrado |
| Versão        | 1               | Versão do formato de cabeçalho (ex: `1`)                                  |
| Algoritmo     | 1               | `1` para AES (reservado para futuras extensões com outros algoritmos)     |
| Modo          | 1               | `1` para modo CBC                                                         |
| IV            | 16              | Vetor de inicialização (gerado aleatoriamente na criptografia)           |
| Reserved      | 11              | Reservado para uso futuro (preencher com `0x00`)                          |

---

### 2. Etapas do Programa (Encrypt)

1. Solicitar ao usuário o caminho de um arquivo para criptografar.  
2. Gerar uma chave de 256 bits (pode ser fixa ou pedida ao usuário).  
3. Gerar o IV (Initialization Vector) aleatoriamente.  
4. Criar o cabeçalho conforme especificado.  
5. Criptografar o conteúdo do arquivo usando AES-CBC.  
6. Escrever o cabeçalho + dados criptografados em um novo arquivo.
7. Salvar o arquivo concatenando ".enc" no final do nome do arquivo.

---

### 3. Descriptografia (Decrypt)

1. Ler o cabeçalho do arquivo cifrado.  
2. Verificar se o identificador, versão e algoritmo estão corretos (validação).  
3. Extrair o IV.  
4. Usar a chave (a mesma da criptografia) para decifrar o conteúdo.  
5. Salvar o arquivo original.

---

## Dicas

- Use o pacote `cryptography` com `Cipher`, `algorithms.AES`, `modes.CBC` e `padding.PKCS7`.
- Lembre-se de aplicar e remover o padding adequadamente.
- Para escrever e ler os campos binários, use a biblioteca `struct` ou concatene os bytes com cuidado.


In [ ]:
#Bibliotecas
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
import secrets

In [ ]:
#CRIPTOGRAFIA AES
def encrypt_aes(key: bytes, iv: bytes, plaintext: bytes) -> bytes:
    """
    Criptografa o texto usando AES no modo CBC.

    :param key: Chave de criptografia de 16, 24 ou 32 bytes.
    :param iv: Vetor de inicialização de 16 bytes.
    :param plaintext: Texto em claro a ser criptografado.
    :return: Texto cifrado.
    """
    # Criação do cifrador AES no modo CBC
    cipher = Cipher(
        algorithms.AES(key),
        modes.CBC(iv),
        backend=default_backend()
    )

    # Criando o objeto de criptografia
    encryptor = cipher.encryptor()

    # Preenchimento do texto em claro para ajustar ao tamanho do bloco
    padder = padding.PKCS7(algorithms.AES.block_size).padder()
    padded_plaintext = padder.update(plaintext) + padder.finalize()

    # Criptografando o texto em claro
    ciphertext = encryptor.update(padded_plaintext) + encryptor.finalize()

    return ciphertext

In [ ]:
# DESCRIPTOGRAFIA AES

def decrypt_aes(key: bytes, iv: bytes, ciphertext: bytes) -> bytes:
    """
    Descriptografa o texto cifrado usando AES no modo CBC.

    :param key: Chave de criptografia de 16, 24 ou 32 bytes.
    :param iv: Vetor de inicialização de 16 bytes.
    :param ciphertext: Texto cifrado a ser descriptografado.
    :return: Texto em claro.
    """
    # Criação do cifrador AES no modo CBC
    cipher = Cipher(
        algorithms.AES(key),
        modes.CBC(iv),
        backend=default_backend()
    )

    # Criando o objeto de descriptografia
    decryptor = cipher.decryptor()

    # Descriptografando o texto cifrado
    padded_plaintext = decryptor.update(ciphertext) + decryptor.finalize()

    # Remoção do preenchimento
    unpadder = padding.PKCS7(algorithms.AES.block_size).unpadder()
    plaintext = unpadder.update(padded_plaintext) + unpadder.finalize()

    return plaintext

In [ ]:
#Cria o arquivo
texto = b'Ola Tudo bem?'
with open("arquivoSecreto.txt", "wb") as f:
  f.write(texto)

In [ ]:
def cifrarArquivo(arquivo, chave):

  # Abrir/ler o arquivo
  with open(arquivo, "rb") as f:
    arqBin = f.read()

   # Cifrar
  iv = secrets.token_bytes(16)
  print(iv)
  enc = encrypt_aes(chave, iv, arqBin)

  # Motar o cabeçalho e concatenar
  header = bytearray()
  header += b'ED'
  header += bytes([0x01])  # versao
  header += bytes([0x01])  # algoritmo
  header += bytes([0x01])  # modo CBC
  header += iv
  header += bytes(11)

  completo = header + enc
  # Salvar o arquivo

  with open(arquivo+'.enc', "wb") as f:
    f.write(completo)

  print(arquivo)



########################################################
arquivo = "arquivoSecreto.txt"
chave  = bytes([1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4,5, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8])
cifrarArquivo(arquivo, chave)


b'\xcer\r\x9d{\xcf\xa0\xbe`\xcb\xbeNd\xf2\xc1\xb9'
arquivoSecreto.txt


In [ ]:
def ler_header(arquivo_enc):

    with open(arquivo_enc, "rb") as f:
        dados = f.read()
        print(dados)

    header = dados[:32]
    print(header)
    texto = dados[32:]
    print(texto)

    #campos do header
    identificador = header[:2]
    print(identificador)
    versao = header[2]
    print(versao)
    algoritmo = header[3]
    print(algoritmo)
    modo = header[4]
    print(modo)
    iv = header[5:21]
    print(iv)

    # validações básicas
    if identificador != b'ED':
        print("Arquivo inválido :não é ED")
        return None
    if algoritmo != 1 or modo != 1:
        print("Algoritmo ou modo inválido")
        return None
    # retorna o necessário para decrypt
    return iv, texto

In [ ]:
iv, texto = ler_header("arquivoSecreto.txt.enc")


b'ED\x01\x01\x01\xcer\r\x9d{\xcf\xa0\xbe`\xcb\xbeNd\xf2\xc1\xb9\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00U\x83\xee}\x96\x1e\x87\x01[\xec\xb3\x96\xb8T8\xc2'
b'ED\x01\x01\x01\xcer\r\x9d{\xcf\xa0\xbe`\xcb\xbeNd\xf2\xc1\xb9\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
b'U\x83\xee}\x96\x1e\x87\x01[\xec\xb3\x96\xb8T8\xc2'
b'ED'
1
1
1
b'\xcer\r\x9d{\xcf\xa0\xbe`\xcb\xbeNd\xf2\xc1\xb9'


In [ ]:
descifrado = decrypt_aes(chave,iv,texto)
print(descifrado)

b'Ola Tudo bem?'


# Exercício 2: Gerando Arquivo de Metadados para Verificação de Integridade

**Objetivo:**  
Criar um programa em Python que não altera o arquivo original, mas gera um **arquivo de metadados** (48 bytes) contendo um cabeçalho e um fingerprint de 16 bytes (último bloco do AES‑CBC) para verificar se o arquivo original foi modificado.

---

## Instruções

### 1. Cabeçalho do Arquivo de Metadados (48 bytes)

O arquivo de metadados deve conter exatamente 48 bytes, dispostos assim:

| Campo         | Tamanho (bytes) | Descrição                                                                                 |
|---------------|-----------------|-------------------------------------------------------------------------------------------|
| Identificador | 2               | Sequência fixa, ex: `b'CF'`, para indicar “Crypto Fingerprint”                            |
| Versão        | 1               | Versão do formato (ex: `1`)                                                               |
| Algoritmo     | 1               | `1` para AES                                                                              |
| Modo          | 1               | `1` para CBC                                                                              |
| IV            | 16              | Vetor de inicialização (aleatório)                                                        |
| Fingerprint   | 16              | Último bloco do ciphertext gerado a partir do arquivo original (garante integridade)      |
| Reserved      | 11              | Reservado para uso futuro (preencher com `0x00`)                                          |

---

### 2. Geração do Arquivo de Metadados

1. **Entrada do Usuário**  
   - Solicitar o caminho do arquivo original a ser protegido.

2. **Chave e IV**  
   - Gerar (ou solicitar) uma chave AES de 256 bits.  
   - Gerar um IV aleatório de 16 bytes.

3. **Cálculo do Fingerprint**  
   - Ler todo o conteúdo do arquivo original.  
   - Aplicar AES‑CBC com padding PKCS7 sobre esse conteúdo **somente em memória** (não salvar o ciphertext em disco).  
   - Extrair os **últimos 16 bytes** do ciphertext gerado como fingerprint.

4. **Montagem do Cabeçalho**  
   - Empacotar, em ordem:  
     1. Identificador (2 bytes)  
     2. Versão (1 byte)  
     3. Algoritmo (1 byte)  
     4. Modo (1 byte)  
     5. IV (16 bytes)  
     6. Fingerprint (16 bytes)  
     7. Reserved (11 bytes de `0x00`)

5. **Gravação do Arquivo de Metadados**  
   - Salvar esses 48 bytes em um novo arquivo (por exemplo, com extensão `.meta`).

---

### 3. Verificação de Integridade

1. **Leitura do Arquivo de Metadados**  
   - Extrair cada campo do cabeçalho e validar.

2. **Reprodução do Fingerprint**  
   - Ler o mesmo arquivo original.  
   - Executar AES‑CBC in‑memory com a chave e IV extraídos.  
   - Extrair os últimos 16 bytes do ciphertext.

3. **Comparação**  
   - Se o fingerprint recalculado for **igual** ao fingerprint armazenado no cabeçalho, o arquivo **não foi alterado**.  
   - Caso contrário, sinalizar “Arquivo modificado ou corrompido”.

---

## Dicas
- Use `cryptography` para `Cipher`, `algorithms.AES`, `modes.CBC` e `padding.PKCS7`.  



In [ ]:
# Função que abre o arquivo, criptografa com AES e pega os últimos 16 bytes

def figerprint(arquivo, key, iv):

    with open(arquivo, "rb") as f:
        dados = f.read()

    texto= encrypt_aes(key, iv, dados)

    figer = texto[-16:]
    return figer

In [ ]:
#Criar arquivo de texte

texto = b'Este texto pertence a Gustavo Camargo'
with open("arquivoGustavo.txt", "wb") as f:
  f.write(texto)

In [ ]:
# Metadados

arquivo = input("Digite o arquivo: ")
chave = bytes([1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4,5, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8])
iv = secrets.token_bytes(16)

Digite o arquivo: arquivoGustavo.txt


In [ ]:
def criarMeta(iv, arquivo, chave):
    # Define campos
  IDENT       = b'CF'                 # 2 bytes
  VERSION     = bytes([0x01])         # 1 byte
  ALGO        = bytes([0x01])         # 1 byte (AES)
  MODE        = bytes([0x01])         # 1 byte (CBC)
  IV          = iv     # 16 bytes: 0x00..0x0F
  FINGERPRINT = figerprint(arquivo,chave, iv)
  RESERVED    = bytes(11)             # 11 bytes de 0x00

  # Monta o header (48 bytes)
  header = bytearray()
  header += IDENT
  header += VERSION
  header += ALGO
  header += MODE
  header += IV
  header += FINGERPRINT
  header += RESERVED

  # Salva no arquivo
  with open("arquivo.meta", "wb") as f:
      f.write(header)


In [ ]:
def comparaFingerPrint(arquivo_original, arquivo_meta, chave, iv):

   finger_arquivo_original = figerprint(arquivo_original,chave,iv)
   print(finger_arquivo_original)


   with open(arquivo_meta, "rb") as f:
    meta = f.read()

   finger_arquivo_meta = meta[21:37]
   print(finger_arquivo_meta)

   if(finger_arquivo_original == finger_arquivo_meta):
    print("tudo safo no arquivo!!!")
   else:
    print("Algo de errado não está certo")


In [ ]:
criarMeta(iv, "arquivoGustavo.txt", chave)

comparaFingerPrint("arquivoGustavo.txt", "arquivo.meta", chave, iv)

b'Q\xe4\xb6\x7f\x15\xe39\x80\xb1\xa6-\x9b\x1b\xa8[\xef'
b'Q\xe4\xb6\x7f\x15\xe39\x80\xb1\xa6-\x9b\x1b\xa8[\xef'
tudo safo no arquivo!!!


In [ ]:
texto = "Corrompendo"
with open("arquivoGustavo.txt", "w") as f:
  f.write(texto)

In [ ]:

comparaFingerPrint("arquivoGustavo.txt", "arquivo.meta", chave, iv)

b'\x1b\xcd\xdc\xdcn\x8f|\xd6\xc0\x80\xe7\x04\x03\xd1D\xbc'
b'Q\xe4\xb6\x7f\x15\xe39\x80\xb1\xa6-\x9b\x1b\xa8[\xef'
Algo de errado não está certo


# Quebrando a banca

Trabalho – Números Pseudoaleatórios

Operação: Quebra-Banca (SORTE-BET)

As casas de apostas vendem a ilusão da sorte, mas entregam apenas algoritmos.
Você teve acesso ao código-fonte da SORTE-BET e pode expor a farsa: a “sorte”
deles tem um rastro previsível. Se o código é determinístico, a banca não é soberana.Hoje, você deve provar que o segredo deles pode ser calculado por qualquer um que saiba o mínimo de segurança da informação.

A Missão: Sua tarefa é demonstrar que o motor de jogo da SORTE-BET é vulnerável à análise técnica. Utilizando o código-fonte extraído e as informações que o servidor teentrega, você deve construir uma ferramenta de previsão capaz de enxergar as cartas
ocultas da casa (o adversário).Na solução crie uma variável chamada betQuebrada.

Tarefas
- Conexão: Acesse o endpoint /iniciar/{seu_nome} e receba as suas 9 cartas (o
servidor rodará em uma máquina do professor).
- Sincronização: Analise o código-fonte para identificar como o estado inicial
do jogo é definido.
- Exploração: Seu script deve processar os dados e imprimir exatamente as 9
cartas que o servidor mantém escondidas na mão dele ("mao_da_casa") antes
de qualquer conferência.
- Quebre a Banca: Valide sua previsão chamando o endpoint
/conferir_resultado/{seu_nome}.
Se os dados baterem, você quebrou a banca!
Dicas:
- Estude o comando time.time()
- Entenda totalmente o que a função obter_semente() retorna.
Entrega:
• Código fonte organizado e comentado;
• A arguição do código também será avaliada.
(Extra): Corrija a vulnerabilidade e tente quebrar novamente.

## servidorJogo.py:

In [1]:
from fastapi import FastAPI, HTTPException
import random
import time

app = FastAPI()

# Banco de dados em memória para gerenciar as sessões
db_sessoes = {}

def obter_semente():
    return int(time.time() * 1000)

@app.get("/iniciar/{nome_jogador}")
def iniciar(nome_jogador: str):
    # Identifica se o jogador já tinha um jogo e avisa sobre o reinício
    aviso = None
    if nome_jogador in db_sessoes:
        aviso = f"Jogo reiniciado para o jogador {nome_jogador}."


    semente = obter_semente()
    random.seed(semente)

    # Estrutura do Baralho Padrão
    naipes = ['O', 'E', 'C', 'P'] # O(uro)  E(spadas) C(opa) P(aus)
    valores = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
    baralho = [f"{v}{n}" for v in valores for n in naipes]

    # Embaralhamento determinístico
    random.shuffle(baralho)

    # Distribuição: 9 cartas para o jogador
    # E 9 cartas ocultas para a casa 9
    mao_jogador = [baralho.pop(0) for _ in range(9)]
    mao_casa = [baralho.pop(0) for _ in range(9)]

    # Salva o estado da partida do jogador
    db_sessoes[nome_jogador] = {
        "jogador": mao_jogador,
        "casa": mao_casa,
        "ativa": True
    }

    return {
        "status": aviso or "Partida Iniciada",
        "jogador": nome_jogador,
        "suas_cartas": mao_jogador
    }

@app.get("/conferir_resultado/{nome_jogador}")
def finalizar(nome_jogador: str):
    if nome_jogador not in db_sessoes:
        raise HTTPException(status_code=404, detail="Jogador não encontrado. Inicie um jogo primeiro.")

    # Recupera a mão da casa e finaliza a partida
    resultado = db_sessoes[nome_jogador]["casa"]
    del db_sessoes[nome_jogador] # Finaliza a sessão

    return {
        "mensagem": f"Partida de {nome_jogador} finalizada.",
        "mao_da_casa": resultado
    }

## QuebrandoBanca.py:

In [ ]:
import requests
import time
import random

NOME = "Marco_Antonio_Nitsche"
URL_BASE = "http://127.0.0.1:8000"

def gerar_baralho():
    naipes = ['O', 'E', 'C', 'P']
    valores = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
    return [f"{v}{n}" for v in valores for n in naipes]

def prever_mao_casa(minhas_cartas, tempo_aprox):
    # janela de busca (em milissegundos)
    for delta in range(-5000, 5000):
        semente = int((tempo_aprox * 1000) + delta)
        random.seed(semente)


        baralho = gerar_baralho()
        random.shuffle(baralho)

        jogador = [baralho.pop(0) for _ in range(9)]

        if jogador == minhas_cartas:
            casa = [baralho.pop(0) for _ in range(9)]
            return casa, semente

    return None, None

def quebrar_banca():
    global betQuebrada

    print(f"[*] Iniciando partida para: {NOME}...")
    tempo_antes = time.time()

    res = requests.get(f"{URL_BASE}/iniciar/{NOME}")
    dados = res.json()

    minhas_cartas = dados["suas_cartas"]
    print(f"[+] Minhas cartas: {minhas_cartas}")

    tempo_depois = time.time()
    tempo_aprox = (tempo_antes + tempo_depois) / 2

    print("[*] Tentando prever a mão da casa...")

    casa_prevista, semente = prever_mao_casa(minhas_cartas, tempo_aprox)

    if casa_prevista:
        print(f"[+] Semente encontrada: {semente}")
        print(f"[+] Mão prevista da casa: {casa_prevista}")

        betQuebrada = casa_prevista

        # Conferindo
        res_final = requests.get(f"{URL_BASE}/conferir_resultado/{NOME}")
        real = res_final.json()["mao_da_casa"]

        print("\n----- RESULTADO -----")
        print(f"Previsto: {casa_prevista}")
        print(f"Real:     {real}")

        if casa_prevista == real:
            print("\nBANCA QUEBRADA!")
        else:
            print("\nAlgo deu errado")

    else:
        print("[-] Não foi possível encontrar a semente")

if __name__ == "__main__":
    quebrar_banca()

Instalar bibliotecas:

```
pip install fastapi uvicorn requests
```

### Como rodar os arquivos?

- Salva os dois aquivos em uma mesma pasta
- Roda primeiro o servidorJogo, no terminal com:

  ```
  uvicorn servidorJogo:app --reload
  ```

- Abre outro terminal e executa:

  ```
  python QuebrandoBanca.py
  ```

### Como desbrir a forma para quebrar a banca?

Em termos técnicos, o uso de timestamp como semente reduz drasticamente a entropia do sistema, tornando o espaço de busca pequeno e previsível. Como o gerador é determinístico, basta testar valores próximos ao tempo conhecido para reproduzir toda a sequência pseudoaleatória.

A semente sempre vai ser a quantidade exata de segundos que se passaram desde 01/01/1970 até o momento que roda o código QuebraBanca.py

Vai contra o principio da **Entropia**, que  mede a aleatoriedade e imprevisibilidade dos dados.

### Como corrigir a falha?

Usando ***secrets***. A diferença é que o random é determinístico e reproduzível a partir de uma seed, enquanto o secrets utiliza fontes imprevisíveis do sistema, garantindo aleatoriedade criptograficamente segura.

random:

“dado o mesmo começo, sempre gera o mesmo resultado”

secrets:

“mesmo que você tente repetir, não consegue gerar igual”

ServidorJogoAjustado.py:

In [5]:
from fastapi import FastAPI, HTTPException
import random
import time
import secrets

app = FastAPI()

# Banco de dados em memória para gerenciar as sessões
db_sessoes = {}

def obter_semente():
    return int(time.time() * 1000)

@app.get("/iniciar/{nome_jogador}")
def iniciar(nome_jogador: str):
    # Identifica se o jogador já tinha um jogo e avisa sobre o reinício
    aviso = None
    if nome_jogador in db_sessoes:
        aviso = f"Jogo reiniciado para o jogador {nome_jogador}."


    semente = obter_semente()
    random.seed(semente)

    # Estrutura do Baralho Padrão
    naipes = ['O', 'E', 'C', 'P'] # O(uro)  E(spadas) C(opa) P(aus)
    valores = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
    baralho = [f"{v}{n}" for v in valores for n in naipes]

    # Embaralhamento que não é mais determinístico
    secrets.SystemRandom().shuffle(baralho)

    # Distribuição: 9 cartas para o jogador
    # E 9 cartas ocultas para a casa 9
    mao_jogador = [baralho.pop(0) for _ in range(9)]
    mao_casa = [baralho.pop(0) for _ in range(9)]

    # Salva o estado da partida do jogador
    db_sessoes[nome_jogador] = {
        "jogador": mao_jogador,
        "casa": mao_casa,
        "ativa": True
    }

    return {
        "status": aviso or "Partida Iniciada",
        "jogador": nome_jogador,
        "suas_cartas": mao_jogador
    }

@app.get("/conferir_resultado/{nome_jogador}")
def finalizar(nome_jogador: str):
    if nome_jogador not in db_sessoes:
        raise HTTPException(status_code=404, detail="Jogador não encontrado. Inicie um jogo primeiro.")

    # Recupera a mão da casa e finaliza a partida
    resultado = db_sessoes[nome_jogador]["casa"]
    del db_sessoes[nome_jogador] # Finaliza a sessão

    return {
        "mensagem": f"Partida de {nome_jogador} finalizada.",
        "mao_da_casa": resultado
    }